# Making Caption

Notebook nay tao caption theo tung shot tu keyframe, subtitle va media-info. Ket qua duoc luu vao `data/caption/{video_id}.json`.

In [1]:
# Neu moi tao moi moi truong, chay dong nay truoc:
# %pip install requests python-dotenv Pillow tqdm pandas

In [2]:
import base64
import csv
import io
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from dotenv import load_dotenv
from PIL import Image
from tqdm.auto import tqdm

load_dotenv()

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "L30_data"

def first_existing_dir(*names):
    for name in names:
        path = DATA_DIR / name
        if path.exists():
            return path
    return DATA_DIR / names[0]

KEYFRAME_DIR = first_existing_dir("keyframes", "keyframe")
MAP_KEYFRAME_DIR = first_existing_dir("map-keyframes", "map_keyframe")
SHOT_DIR = first_existing_dir("shots", "shot")
SUBTITLE_DIR = first_existing_dir("subtitles", "subtitle")
MEDIA_INFO_DIR = first_existing_dir("media-info", "media_info")
CAPTION_DIR = first_existing_dir("captions", "caption")

# Dien danh sach video can chay, vi du: ["L21_V001", "L21_V002"]. De [] thi tu quet data/shot.
VIDEO_IDS = ["L30_V001", "L30_V002", "L30_V003", "L30_V004", "L30_V005", "L30_V006", "L30_V007", "L30_V008", "L30_V009", "L30_V010", "L30_V011", "L30_V012", "L30_V013", "L30_V014", "L30_V015", "L30_V016", "L30_V017", "L30_V018", "L30_V019", "L30_V020", "L30_V021", "L30_V022", "L30_V023", "L30_V024", "L30_V025", "L30_V026", "L30_V027", "L30_V028", "L30_V029", "L30_V030", "L30_V031", "L30_V032", "L30_V033", "L30_V034", "L30_V035", "L30_V036", "L30_V037", "L30_V038", "L30_V039", "L30_V040", "L30_V041", "L30_V042", "L30_V043", "L30_V044", "L30_V045", "L30_V046", "L30_V047", "L30_V048", "L30_V049", "L30_V050", "L30_V051", "L30_V052", "L30_V053", "L30_V054", "L30_V055", "L30_V056", "L30_V057", "L30_V058", "L30_V059", "L30_V060", "L30_V061", "L30_V062", "L30_V063", "L30_V064", "L30_V065", "L30_V066", "L30_V067", "L30_V068", "L30_V069", "L30_V070", "L30_V071", "L30_V072", "L30_V073", "L30_V074", "L30_V075", "L30_V076", "L30_V077", "L30_V078", "L30_V079", "L30_V080", "L30_V081", "L30_V082", "L30_V083", "L30_V084", "L30_V085", "L30_V086", "L30_V087", "L30_V088", "L30_V089", "L30_V090", "L30_V091", "L30_V092", "L30_V093", "L30_V094", "L30_V095", "L30_V096"]
# De test nhanh truoc khi chay hang loat.
MAX_VIDEOS = None
MAX_SHOTS_PER_VIDEO = None

MODEL = os.getenv("RECAP_MODEL", "gpt-5.4-mini")
API_BASE = os.getenv("RECAP_API_BASE", "https://api.apimaster.ai/v1/chat/completions")
API_KEY = os.getenv("RECAP_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY")
MAX_WORKERS = 1
KEYFRAMES_PER_SHOT = 1
OVERWRITE = False

CAPTION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base dir: {BASE_DIR}")
print(f"Output caption: {CAPTION_DIR}")
print(f"API base: {API_BASE}")
print(f"Model: {MODEL}")

Base dir: d:\Truong_pipeline\My_Code
Output caption: d:\Truong_pipeline\My_Code\L30_data\captions
API base: https://api.apimaster.ai/v1/chat/completions
Model: gpt-5.4-mini


d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SYSTEM_PROMPT = """
You are an advanced Large Vision-Language Model generating temporally consistent shot-level captions and updating recurrent memory for a video retrieval system.

The dataset consists of videos from the "Lan tỏa năng lượng tích cực 2024" contest, featuring human-interest stories, overcoming hardship, charity, community support, and healthy/active lifestyles.

## Inputs
1. Video Info: title, description, and overall story.
2. Previous Memory (M_{t-1}): contextual information accumulated from previous shots.
3. Subtitle (S_t): transcript/voiceover of the current shot.
4. Keyframes: images representing the CURRENT shot.

## Task

### 1. Analyze the Current Shot
Use both keyframes and subtitle to understand the current shot.

Evidence priority:
**Keyframes > Current Subtitle > Previous Memory**

Keyframes are the primary evidence for visual content. Only describe people, objects, actions, locations, clothing, or environments that are visibly supported by the keyframes.

Use the subtitle for spoken facts, names, locations, occupations, events, and background information.

Use Previous Memory only to maintain continuity, resolve identities, and interpret references. Never assume that an object or activity from Previous Memory is still present unless it is visible or mentioned in the current shot.

Do not hallucinate or invent unsupported details.

### 2. Update Memory (M_t)

Update Previous Memory using information from the CURRENT shot.

Memory structure:

#Program:
#Topic:
#Location:
#People:
#Activities_Props:

Keep stable information such as the program, main topic, and established character identities.

For #Location, describe the CURRENT physical location/setting only when it is visible or explicitly identified in the current shot. Do not use a character's residence or a historical location as the current location.

For #People, preserve established identities and merge an unidentified person into a named character when the current shot provides enough evidence. Do not create duplicate identities.

### STRICT DYNAMIC MEMORY PRUNING

#Activities_Props contains only temporary/current information such as objects, equipment, food, donated goods, tools, or activities.

Keep a dynamic item ONLY if it is:
1. visibly present in the current keyframes, OR
2. explicitly mentioned in the current subtitle, OR
3. necessary to interpret an explicit reference in the current subtitle.

Otherwise DELETE it.

Do NOT copy dynamic items from Previous Memory simply because they appeared in earlier shots.

Mark recurring people/items with "(previous focus)" or "(seen earlier)" when appropriate.

### 3. Generate Caption (C_t)

Write ONE concise, factual paragraph describing the CURRENT shot.

Prioritize:
1. Visible content and actions
2. Important information from the current subtitle
3. Relevant context from Previous Memory

Describe the shot directly. Do not say:
"The subtitle explains..."
"The narration states..."
"The previous memory..."

Do not summarize the entire video or add emotional fluff, speculation, or unsupported details.

The caption must be in English, while preserving Vietnamese proper names, organization names, locations, dish names, and specific titles exactly as provided.

## Output

Respond strictly in valid JSON with no Markdown:

{
  "memory": "updated memory string formatted exactly like the input memory structure",
  "caption": "concise factual paragraph describing the current shot in English"
}
""".strip()

In [4]:
def load_json(path, default=None):
    if not Path(path).exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def discover_video_ids():
    ids = sorted(p.stem for p in SHOT_DIR.glob("*.json"))
    if VIDEO_IDS:
        ids = [vid for vid in VIDEO_IDS if (SHOT_DIR / f"{vid}.json").exists()]
    if MAX_VIDEOS is not None:
        ids = ids[:MAX_VIDEOS]
    return ids


def get_subtitle_for_shot(shot_start, shot_end, subtitles):
    texts = []
    for sub in subtitles or []:
        if float(sub.get("start", 0)) <= shot_end and float(sub.get("end", 0)) >= shot_start:
            text = str(sub.get("text", "")).strip()
            if text:
                texts.append(text)
    return " ".join(texts)


def format_video_info(media_info):
    if not media_info:
        return ""
    fields = {
        "Title": media_info.get("title", ""),
        "Author": media_info.get("author", ""),
    }
    return "\n".join(f"{key}: {value}" for key, value in fields.items() if value)


def image_candidates(video_id, n_value):
    n_int = int(float(n_value))
    names = [
        f"{n_int:06d}.jpg", f"{n_int:05d}.jpg", f"{n_int:04d}.jpg", f"{n_int:03d}.jpg", f"{n_int}.jpg",
        f"{n_int:06d}.png", f"{n_int:05d}.png", f"{n_int:04d}.png", f"{n_int:03d}.png", f"{n_int}.png",
    ]
    return [KEYFRAME_DIR / video_id / name for name in names]


def load_keyframe_map(video_id):
    path = MAP_KEYFRAME_DIR / f"{video_id}.csv"
    rows = []
    if not path.exists():
        return rows
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            try:
                rows.append({
                    "n": int(float(row.get("n", 0))),
                    "pts_time": float(row.get("pts_time", 0)),
                    "fps": float(row.get("fps", 0)) if row.get("fps") not in (None, "") else None,
                    "frame_idx": int(float(row.get("frame_idx", 0))) if row.get("frame_idx") not in (None, "") else None,
                })
            except ValueError:
                continue
    return rows


def pick_keyframes_for_shot(video_id, shot, keyframe_rows, max_images=1):
    start = float(shot.get("start_time", 0))
    end = float(shot.get("end_time", start))
    midpoint = (start + end) / 2
    selected = [row for row in keyframe_rows if start <= row["pts_time"] <= end]
    if not selected and keyframe_rows:
        selected = [min(keyframe_rows, key=lambda row: abs(row["pts_time"] - midpoint))]
    if not selected:
        return []

    if max_images <= 1:
        chosen = [min(selected, key=lambda row: abs(row["pts_time"] - midpoint))]
    else:
        step = max(1, len(selected) // max_images)
        chosen = selected[::step][:max_images]

    paths = []
    for row in chosen:
        for candidate in image_candidates(video_id, row["n"]):
            if candidate.exists():
                paths.append(candidate)
                break
    return paths

In [5]:
def encode_image(image_path, max_size=(512, 512)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail(max_size)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def extract_json_object(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise


def call_recap_api(video_info, keyframe_paths, subtitle, previous_memory, model=MODEL):
    if not API_KEY:
        raise RuntimeError("Chua co RECAP_API_KEY hoac OPENAI_API_KEY trong bien moi truong/file .env")

    content = [{
        "type": "text",
        "text": (
            f"Video Info:\n{video_info}\n\n"
            f"Previous Memory:\n{previous_memory}\n\n"
            f"Subtitle:\n{subtitle}\n\n"
            "Keyframes:"
        ),
    }]

    for image_path in keyframe_paths:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
        })

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        "response_format": {"type": "json_object"},
        "max_tokens": 1000,
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(5):
        try:
            response = requests.post(
                API_BASE,
                headers=headers,
                json=payload,
                timeout=90,
            )
            if response.status_code in {429, 502, 503, 504}:
                wait_sec = 3 ** attempt
                print(f"API HTTP {response.status_code}, thu lai sau {wait_sec}s")
                time.sleep(wait_sec)
                continue
            response.raise_for_status()
            message = response.json()["choices"][0]["message"]["content"]
            result = extract_json_object(message)
            return {
                "memory": str(result.get("memory", previous_memory)).strip(),
                "caption": str(result.get("caption", "")).strip(),
            }
        except Exception as exc:
            if attempt == 4:
                raise
            wait_sec = 2 + attempt * 3
            print(f"Loi API: {exc}. Thu lai sau {wait_sec}s")
            time.sleep(wait_sec)

In [6]:
def process_video(video_id):
    output_path = CAPTION_DIR / f"{video_id}.json"
    partial_path = CAPTION_DIR / f"{video_id}.partial.json"
    if output_path.exists() and not OVERWRITE:
        return {"video_id": video_id, "status": "skipped", "path": str(output_path), "shots": None}

    shots = load_json(SHOT_DIR / f"{video_id}.json", default=[])
    subtitles = load_json(SUBTITLE_DIR / f"{video_id}.json", default=[])
    media_info = load_json(MEDIA_INFO_DIR / f"{video_id}.json", default={})
    keyframe_rows = load_keyframe_map(video_id)

    if MAX_SHOTS_PER_VIDEO is not None:
        shots = shots[:MAX_SHOTS_PER_VIDEO]

    video_info = format_video_info(media_info)
    previous_memory = "None"
    output = []

    print(f"\n===== {video_id}: {len(shots)} shots =====")
    for shot in tqdm(shots, desc=video_id):
        start = float(shot.get("start_time", 0))
        end = float(shot.get("end_time", start))
        subtitle = get_subtitle_for_shot(start, end, subtitles)
        keyframes = pick_keyframes_for_shot(video_id, shot, keyframe_rows, KEYFRAMES_PER_SHOT)

        result = call_recap_api(video_info, keyframes, subtitle, previous_memory)
        previous_memory = result["memory"] or previous_memory

        output.append({
            "shot_id": shot.get("shot_id"),
            "start_time": start,
            "end_time": end,
            "duration": round(end - start, 3),
            "keyframes": [str(p.relative_to(BASE_DIR)) for p in keyframes],
            "subtitle": subtitle,
            "caption": result["caption"],
            "memory": previous_memory,
        })
        save_json(partial_path, output)

    save_json(output_path, output)
    if partial_path.exists():
        partial_path.unlink()
    return {"video_id": video_id, "status": "done", "path": str(output_path), "shots": len(output)}

In [7]:
video_ids = discover_video_ids()
print(f"Tong so video se xu ly: {len(video_ids)}")

if not video_ids:
    raise FileNotFoundError(f"Khong tim thay file shot nao trong {SHOT_DIR}")

if MAX_WORKERS <= 1:
    results = [process_video(video_id) for video_id in video_ids]
else:
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_video, video_id) for video_id in video_ids]
        for future in as_completed(futures):
            results.append(future.result())

results

Tong so video se xu ly: 96

===== L30_V001: 6 shots =====


L30_V001:   0%|          | 0/6 [00:00<?, ?it/s]

L30_V001: 100%|██████████| 6/6 [01:02<00:00, 10.39s/it]



===== L30_V002: 7 shots =====


L30_V002: 100%|██████████| 7/7 [00:47<00:00,  6.82s/it]


...


===== L30_V095: 13 shots =====


L30_V095: 100%|██████████| 13/13 [01:14<00:00,  5.73s/it]



===== L30_V096: 4 shots =====


L30_V096: 100%|██████████| 4/4 [00:24<00:00,  6.02s/it]


[{'video_id': 'L30_V001',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V001.json',
  'shots': 6},
 {'video_id': 'L30_V002',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V002.json',
  'shots': 7},
 {'video_id': 'L30_V003',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V003.json',
  'shots': 14},
 {'video_id': 'L30_V004',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V004.json',
  'shots': 4},
 {'video_id': 'L30_V005',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V005.json',
  'shots': 3},
 {'video_id': 'L30_V006',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V006.json',
  'shots': 7},
 {'video_id': 'L30_V007',
  'status': 'done',
  'path': 'd:\\Truong_pipeline\\My_Code\\L30_data\\captions\\L30_V007.json',
  'shots': 13},
 {'video_id': 'L30_V008',
  'sta

In [8]:
# Xem nhanh mot file ket qua.
sample_outputs = sorted(CAPTION_DIR.glob("*.json"))
if sample_outputs:
    sample = load_json(sample_outputs[0], default=[])
    print(sample_outputs[0])
    print(json.dumps(sample[:2], ensure_ascii=False, indent=2))
else:
    print("Chua co file caption nao.")

d:\Truong_pipeline\My_Code\L30_data\captions\L30_V001.json
[
  {
    "shot_id": 1,
    "start_time": 0.0,
    "end_time": 26.16,
    "duration": 26.16,
    "keyframes": [
      "L30_data\\keyframes\\L30_V001\\004.jpg"
    ],
    "subtitle": "Đây là một số chiếc xe mà mình nhận được các mảnh thường quân về, một số thì đã sửa xong rồi. Sau khi hoàn thiện thì mình sẽ gửi tặng cho các bạn học sinh ở các vùng sâu, vùng xa có hoàn cảnh đặc biệt khó khăn.",
    "caption": "A bald man bends over a bicycle in a narrow corridor beside a building, working on repairs with other bicycle frames and parts nearby. The shot shows donated bicycles being restored so they can later be given to students in remote areas with especially difficult circumstances.",
    "memory": "#Program: Lan tỏa năng lượng tích cực 2024\n#Topic: A man restores donated bicycles to give to disadvantaged students.\n#Location: A narrow outdoor corridor or alley beside a building, with stairs in the background.\n#People: A bald m